# 29 - External-Mapped Fine-Tuned Retrieval Metrics

Evaluates only the new external-mapped fine-tuned embedding/reranker outputs. Existing base and previous tuned metrics are kept in the earlier metric ledger and are not recomputed here.

In [ ]:
!python -m pip install -q -U "sentence-transformers>=5.1.0" "transformers>=4.51.0" accelerate peft bitsandbytes "faiss-cpu>=1.8.0" rank-bm25 tqdm
!python -m pip uninstall -y torchao
import faiss
print('faiss ok:', faiss.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import sys
import torch

DRIVE_ROOT = Path('/content/drive/MyDrive/TURKISH_LEGAL_RAG')
sys.path.insert(0, str(DRIVE_ROOT))
sys.path.insert(0, str(DRIVE_ROOT / 'src'))
import os
os.chdir(DRIVE_ROOT)
print('Working directory:', Path.cwd())

config = json.loads((DRIVE_ROOT / 'project_config.json').read_text(encoding='utf-8'))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
print('Project:', DRIVE_ROOT)

In [ ]:
benchmark_csv = DRIVE_ROOT / config['benchmark_csv']
corpus_csv = DRIVE_ROOT / config['main_law_corpus_csv']
base_index_root = DRIVE_ROOT / config['best_retrieval']['index_root']

tuned_embedding_model = DRIVE_ROOT / 'models/embedding_tuned/qwen3_embedding_8b_external_mapped_lora_v1'
tuned_embedding_index_root = DRIVE_ROOT / 'indexes/official_law_v3_qwen3_embedding_8b_external_mapped_lora_v1'
tuned_reranker_model = DRIVE_ROOT / 'models/reranker_tuned/qwen3_reranker_8b_external_mapped_lora_v1'

output_dir = DRIVE_ROOT / 'outputs/external_mapped_ablation'
output_dir.mkdir(parents=True, exist_ok=True)

for required in [benchmark_csv, corpus_csv, tuned_embedding_model, tuned_reranker_model]:
    if not Path(required).exists():
        raise FileNotFoundError(required)
if not (base_index_root / 'index_manifest.json').exists():
    raise FileNotFoundError(f'Base index not found: {base_index_root}')

print('Benchmark:', benchmark_csv)
print('Base index:', base_index_root)
print('Tuned embedding:', tuned_embedding_model)
print('Tuned reranker:', tuned_reranker_model)

In [ ]:
from src.build_index import build_indexes

manifest = build_indexes(
    corpus_path=corpus_csv,
    index_root=tuned_embedding_index_root,
    embedding_model=str(tuned_embedding_model),
    text_field=config.get('retrieval_text_field', 'retrieval_text'),
    batch_size=4,
    device=device,
)
print(json.dumps({
    'index_root': str(tuned_embedding_index_root),
    'record_count': manifest.get('record_count'),
    'embedding_count': manifest.get('embedding_count'),
    'embedding_dim': manifest.get('embedding_dim'),
}, ensure_ascii=False, indent=2))

In [ ]:
from src.evaluation_retrieval import evaluate_retrieval

dense_summary = evaluate_retrieval(
    benchmark_csv=benchmark_csv,
    index_root=tuned_embedding_index_root,
    output_predictions_csv=output_dir / 'tuned_embedding_dense_only_predictions.csv',
    output_summary_json=output_dir / 'tuned_embedding_dense_only_summary.json',
    mode='dense',
    top_k=10,
    candidate_k=30,
    device=device,
)
print(json.dumps(dense_summary['metrics'], ensure_ascii=False, indent=2))

In [ ]:
from src.evaluation_reranker import evaluate_reranker

tuned_embedding_base_reranker = evaluate_reranker(
    benchmark_csv=benchmark_csv,
    index_root=tuned_embedding_index_root,
    output_predictions_csv=output_dir / 'tuned_embedding_base_reranker_predictions.csv',
    output_summary_json=output_dir / 'tuned_embedding_base_reranker_summary.json',
    candidate_mode='dense',
    candidate_k=30,
    top_k=10,
    reranker_model='Qwen/Qwen3-Reranker-8B',
    batch_size=4,
    device=device,
)
print(json.dumps(tuned_embedding_base_reranker['metrics'], ensure_ascii=False, indent=2))

In [ ]:
base_embedding_tuned_reranker = evaluate_reranker(
    benchmark_csv=benchmark_csv,
    index_root=base_index_root,
    output_predictions_csv=output_dir / 'base_embedding_tuned_reranker_predictions.csv',
    output_summary_json=output_dir / 'base_embedding_tuned_reranker_summary.json',
    candidate_mode='dense',
    candidate_k=30,
    top_k=10,
    reranker_model=str(tuned_reranker_model),
    batch_size=4,
    device=device,
)
print(json.dumps(base_embedding_tuned_reranker['metrics'], ensure_ascii=False, indent=2))

In [ ]:
tuned_embedding_tuned_reranker = evaluate_reranker(
    benchmark_csv=benchmark_csv,
    index_root=tuned_embedding_index_root,
    output_predictions_csv=output_dir / 'tuned_embedding_tuned_reranker_predictions.csv',
    output_summary_json=output_dir / 'tuned_embedding_tuned_reranker_summary.json',
    candidate_mode='dense',
    candidate_k=30,
    top_k=10,
    reranker_model=str(tuned_reranker_model),
    batch_size=4,
    device=device,
)
print(json.dumps(tuned_embedding_tuned_reranker['metrics'], ensure_ascii=False, indent=2))

In [ ]:
import pandas as pd

summary_files = {
    'external_mapped_tuned_embedding_dense_only': output_dir / 'tuned_embedding_dense_only_summary.json',
    'external_mapped_tuned_embedding_base_reranker': output_dir / 'tuned_embedding_base_reranker_summary.json',
    'external_mapped_base_embedding_tuned_reranker': output_dir / 'base_embedding_tuned_reranker_summary.json',
    'external_mapped_tuned_embedding_tuned_reranker': output_dir / 'tuned_embedding_tuned_reranker_summary.json',
}

rows = []
for system_name, path in summary_files.items():
    summary = json.loads(Path(path).read_text(encoding='utf-8'))
    rows.append({'system': system_name, **summary.get('metrics', {})})

df = pd.DataFrame(rows)
cols = ['system','article_hit@5','article_hit@10','article_recall@5','article_recall@10','article_mrr','article_ndcg@5','article_ndcg@10','doc_hit@10','doc_mrr']
cols = [col for col in cols if col in df.columns]
display(df[cols].round(4))

out_path = output_dir / 'external_mapped_finetuned_metric_table.csv'
df.to_csv(out_path, index=False, encoding='utf-8-sig')
print('Saved:', out_path)